# Tricubic Hermite Box Drop with IPC Obstacle — Full Demo

A cubic Hermite box drops under gravity onto a static `bottom.obj`
obstacle via IPC (Incremental Potential Contact).

**Preview** (~80 steps) verifies the pipeline quickly.
Set `RUN_FULL = True` for the full 400-step / 200-frame animation.


In [ ]:
from pathlib import Path

import numpy as np

import pypgo as pgo
import pypgo.contact as pc
import pypgo.energy as pe
import pypgo.fem as pf
import pypgo.sim as psim
import pypgo.solver as ps
from pypgo.animation import dump_mesh_animation
from pypgo.mesh import read_obj


def _find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / ".git").exists():
            return candidate
    return cwd


REPO_ROOT = _find_repo_root()
ASSET_DIR = Path(pgo.__file__).resolve().parent / "examples" / "assets"


In [ ]:
# ── Volume mesh (cubic hex) ──────────────────────────────────
BOX_VEG = ASSET_DIR / "veg" / "cubic" / "box.veg"
volume = pgo.mesh.veg.VolumeMesh.from_veg_file(
    pgo.mesh.veg.read_veg(str(BOX_VEG))
)

# ── Display surface ───────────────────────────────────────────
surface = read_obj(str(ASSET_DIR / "obj" / "box.obj"))

# ── Static obstacle ───────────────────────────────────────────
bottom_mesh = read_obj(str(ASSET_DIR / "obj" / "bottom.obj"))

print("Volume:  ", volume.num_vertices, "verts,",
      volume.num_elements, "cubes", sep="")
print("Surface: ", surface.num_vertices, "verts,",
      surface.num_elements, "tris", sep="")
print("Obstacle:", bottom_mesh.num_vertices, "verts,",
      bottom_mesh.num_elements, "tris", sep="")

gap = surface.vertices[:, 1].min() - bottom_mesh.vertices[:, 1].max()
print("Gap (box bottom to obstacle):", round(gap, 4), "m")
t_fall = np.sqrt(2 * gap / 9.81)
print("Free-fall time to obstacle:   ", round(t_fall, 4), "s")


## 1. Hermite Deformation, Mass, and Gravity


In [ ]:
formulation = pf.TricubicHermite()
sim_mesh = psim.SimulationMesh.create_volumetric(volume)

deformation_state = pf.deformation_model_state(
    sim_mesh,
    elastic=pf.StableNeo(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=0),
    plastic_field=pf.ElementwiseField(),
)
deformation = pf.deformation_energy(
    deformation_state,
    formulation=formulation,
    options=pf.DeformationOptions(enable_material_max_step=False),
)

mass = pf.formulation_mass_matrix(volume, formulation)
gravity = pf.body_force(volume, formulation, [0.0, -9.81, 0.0])

print("Hermite DOFs:", deformation.num_dofs,
      "(", volume.num_vertices, "x 24)", sep="")
print("Mass: ", mass.shape[0], "x", mass.shape[1],
      " nnz=", mass.nnz, sep="")


## 2. Hermite Surface Embedding and IPC Obstacle


In [ ]:
# ── Hermite surface embedding ─────────────────────────────────
W = pf.surface_embedding_matrix(volume, surface.vertices, formulation)
contact_surface = pc.ContactSurface.embedded(surface.vertices, W)
print("W:", W.shape[0], "x", W.shape[1])

# ── Static obstacle ───────────────────────────────────────────
obstacle = pc.ObstacleSpec.static(
    bottom_mesh.vertices, bottom_mesh.elements
)

# ── IPC ───────────────────────────────────────────────────────
IPC_PARAMS = dict(dhat=0.002, dhat_external=0.005, kappa=3000.0)
ipc = pc.IPCEnergy(
    contact_surface,
    surface.elements,
    params=pc.IPCParameters(**IPC_PARAMS),
    obstacles=[obstacle],
)
print("IPC: dhat=", IPC_PARAMS["dhat"],
      " dhat_ext=", IPC_PARAMS["dhat_external"],
      " kappa=", IPC_PARAMS["kappa"], sep="")
print("IPC at rest:", ipc.value(np.zeros(deformation.num_dofs)))


## 3. Dynamic Simulation Setup

- `damping=False` is **critical** for Hermite: Hessian damping adds a
  large λI that kills the rigid-translation mode.
- Full run: 400 steps at dt=0.005 = 2.0 s physical time, matching the
  contact API demo.


In [ ]:
total_energy = pe.EnergySet([
    (deformation, 1.0),
    (ipc, 1.0),
])

state0 = psim.DynamicState(
    displacement=np.zeros(deformation.num_dofs),
    velocity=np.zeros(deformation.num_dofs),
    acceleration=np.zeros(deformation.num_dofs),
)

DT = 0.005
sim = psim.DynamicSimulation(
    mass=mass,
    state=state0,
    timestep=DT,
    energy=total_energy,
    integrator="implicit_euler",
    damping=(0.0, 0.0),
)

optimizer = ps.NewtonOptimizer(
    max_iterations=200,
    gradient_tolerance=1e-4,
    damping=False,
    line_search="backtrack",
    sparse_solver="auto",
)

# ── Run configuration ──────────────────────────────────────────
RUN_FULL = True                      # flip to True for 200-frame export

FULL_STEPS = 400                     # 2.0 s simulation
FULL_DUMP_INTERVAL = 2               # -> 200 frames

PREVIEW_STEPS = 80                   # quick smoke test
PREVIEW_DUMP_INTERVAL = 4            # ->  20 frames

NUM_STEPS = FULL_STEPS if RUN_FULL else PREVIEW_STEPS
DUMP_INTERVAL = FULL_DUMP_INTERVAL if RUN_FULL else PREVIEW_DUMP_INTERVAL

print("dt =", DT, " | integrator = implicit_euler")
print("DOFs =", sim.num_dofs)
print("Newton: damping=False, backtrack, max 200 iters, tol 1e-4")
print()
print("RUN_FULL =", RUN_FULL," ->", NUM_STEPS, "steps, dump every",
      DUMP_INTERVAL, "->", NUM_STEPS // DUMP_INTERVAL, "frames")


## 4. Run Simulation


In [ ]:
surf_disps = []          # dumped frames only (for Alembic)
frame_norms = []
dump_counter = 0

print("Running", NUM_STEPS, "steps ...")
for k in range(NUM_STEPS):
    frame = sim.step(external_force=gravity, optimizer=optimizer)

    surf_disp = (W @ frame.displacement).reshape(-1, 3)
    frame_norms.append(float(np.linalg.norm(frame.displacement)))

    # Dump every DUMP_INTERVAL frames (plus first and last)
    dump_counter += 1
    if dump_counter >= DUMP_INTERVAL or k == 0 or k == NUM_STEPS - 1:
        surf_disps.append(surf_disp.ravel())
        dump_counter = 0

    if (k + 1) % 20 == 0 or k == 0:
        y_min = float((surface.vertices + surf_disp)[:, 1].min())
        obs_y_max = bottom_mesh.vertices[:, 1].max()
        print("  step", k + 1,
              ": t=", round(sim.state.time, 4),
              " ||u||=", round(frame_norms[-1], 4),
              " box_y_min=", round(y_min, 4),
              " gap=", round(y_min - obs_y_max, 4),
              " iters=", frame.solver_result.iterations,
              " ", frame.solver_result.status.name, sep="")

    if not frame.accepted:
        print("Stopping after rejected step", k + 1)
        break

print()
print("Done. ", len(surf_disps), " frames dumped (",
      len(frame_norms), " steps).", sep="")
print("Final ||u|| =", round(frame_norms[-1], 4))


## 5. Results


In [ ]:
final_surf_disp = surf_disps[-1].reshape(-1, 3)
deformed = surface.vertices + final_surf_disp

print("Rest y:    ", surface.vertices[:, 1].min(), "...",
      surface.vertices[:, 1].max())
print("Final y:   ", deformed[:, 1].min(), "...", deformed[:, 1].max())
obs_y_max = bottom_mesh.vertices[:, 1].max()
box_y_min = deformed[:, 1].min()
print("Obstacle:  ", bottom_mesh.vertices[:, 1].min(), "...", obs_y_max)
print()
print("Box-obstacle gap:", box_y_min - obs_y_max)
print("Contact?", "YES" if box_y_min - obs_y_max < IPC_PARAMS["dhat_external"] else "NO")


In [ ]:
import os
out_path = str(REPO_ROOT / "hermite_box_drop.abc")
dump_mesh_animation(
    out_path, "hermite_box_drop",
    rest_positions=surface.vertices.ravel(),
    displacements=surf_disps,
    triangles=surface.elements.ravel(),
)
size_kb = os.path.getsize(out_path) / 1024
print("Alembic:", out_path, "(", round(size_kb, 1), "KB)", sep="")
print("Frames exported:", len(surf_disps))
